# P0 · Benchmark bộ sinh công thức trên Kaggle

Chạy 4 ứng viên (`bench/models.yaml`) trên 76 prompt × 3 seed, lượng tử NF4, 2 card T4 chạy song song.

**Cài đặt notebook trước khi chạy** (panel bên phải → *Session options*):
- **Accelerator: GPU T4 x2**. Không dùng P100: kiến trúc Pascal, bitsandbytes 4-bit không đảm bảo.
- **Internet: On** (cần xác minh số điện thoại tài khoản Kaggle).
- Chạy nền không cần mở trình duyệt: *Save Version → Save & Run All (Commit)*, tối đa 12 giờ.

**Chạy tiếp khi bị ngắt:** mở phiên mới, *Add Input* → chọn output của version trước. Ô số 2 tự chép
`bench/raw/*.jsonl` cũ sang, `run_infer.py` bỏ qua các lượt đã có.

In [1]:
# 1. Cài thư viện. Ghim transformers đúng bản đã thử trên máy local (Gemma-4 cần >= 5.5).
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
!pip install -q -U "transformers==5.5.0" "bitsandbytes>=0.46" accelerate nvidia-ml-py pyyaml
!python -c "import torch, transformers, bitsandbytes; print('torch', torch.__version__, '| transformers', transformers.__version__, '| bnb', bitsandbytes.__version__, '| GPU', torch.cuda.device_count())"

index, name, memory.total [MiB]
0, Tesla T4, 15360 MiB
1, Tesla T4, 15360 MiB
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 108.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 4.1 MB/s eta 0:00:00
torch 2.10.0+cu128 | transformers 5.5.0 | bnb 0.50.2 | GPU 2


In [2]:
# 2. Lấy code + bộ thử từ GitHub, chép kết quả cũ nếu có.
import glob, os, shutil, subprocess

REPO, BRANCH = "https://github.com/hoanganhquanCS04/Nico-tick.git", "quan_dev"
WORK = "/kaggle/working/Nico-tick"
if os.path.isdir(WORK):
    subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO, WORK], check=True)
os.chdir(WORK)
subprocess.run(["git", "log", "-1", "--format=commit %h %s"], check=True)

os.makedirs("bench/raw", exist_ok=True)
old = [p for p in glob.glob("/kaggle/input/**/bench/raw/*", recursive=True) if p.endswith((".jsonl", ".json"))]
for p in old:
    dst = os.path.join("bench/raw", os.path.basename(p))
    if not os.path.exists(dst) or os.path.getsize(p) > os.path.getsize(dst):
        shutil.copy(p, dst)
print(f"Chép {len(old)} file kết quả cũ" if old else "Chạy mới từ đầu")

# Trọng số mô hình (~19GB) để ở ổ tạm, không để trong /kaggle/working (giới hạn 20GB, bị lưu làm output)
os.environ["HF_HOME"] = "/tmp/hf"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["PYTHONWARNINGS"] = "ignore"

Cloning into '/kaggle/working/Nico-tick'...


commit 9d8b74c Update prompts preview and adjust character limits in prompt generation
Chạy mới từ đầu


In [3]:
# 3. Chạy thử: mỗi mô hình 2 prompt x 1 seed, ghi vào bench/smoke (không lẫn với kết quả thật).
#    Mục đích: bắt lỗi nạp mô hình / fp16 tràn số (Gemma trên T4) TRƯỚC khi chạy 4-6 tiếng.
SMOKE = ["qwen35-0.8b", "sailor2-1b", "qwen35-2b", "gemma4-e2b-plecpu"]
for key in SMOKE:
    !CUDA_VISIBLE_DEVICES=0 python bench/run_infer.py --model {key} --seeds 0 --ids F1-001 F2-001 --out bench/smoke
!python bench/score.py --raw bench/smoke --out bench/smoke

[qwen35-0.8b] Qwen/Qwen3.5-0.8B@2fc06364 | 2 prompt x seed [0] | còn 2 lượt
The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
  nạp xong 15.4s | VRAM card: trước 451 MB -> sau nạp 1349 MB | torch 738 MB | đo bằng nvml
  seed 0 F2-001  2/2 | JSON ok 50% | 32.042s 14.64 tok/s | VRAM 1791.2 MB | còn ~0 phút
[qwen35-0.8b] xong 2 lượt trong 1.1 phút -> bench/smoke
[sailor2-1b] sail/Sailor2-1B-Chat@51b48ecd | 2 prompt x seed [0] | còn 2 lượt
  nạp xong 11.2s | VRAM card: trước 451 MB -> sau nạp 1487 MB | torch 873 MB | đo bằng nvml
  seed 0 F2-001  2/2 | JSON ok 0% | 20.132s 11.08 tok/s | VRAM 2067.2 MB | còn ~0 phút
[sailor2-1b] xong 2 lượt trong 1.1 phút -> bench/smoke
[qwen35-2b] Qwen/Qwen3.5-2B@15852e8c | 2 prompt x seed [0] | còn 2 lượt
The fast path is not available because one of the req

In [4]:
# 3b. Soi output thô của lần chạy thử. Gemma ra chuỗi rỗng / ký tự lặp vô nghĩa = fp16 tràn số
#     -> sửa compute_dtype của gemma trong bench/models.yaml thành float32 rồi chạy lại ô 3.
import json
for p in sorted(glob.glob("bench/smoke/*__seed0.jsonl")):
    for ln in open(p, encoding="utf-8"):
        r = json.loads(ln)
        print(f"== {r['model']} {r['prompt_id']} | {r.get('n_gen_tokens')} token | {r.get('latency_s')}s | VRAM {r.get('vram_nvml_peak_mb')} MB")
        print((r.get("raw_output") or r.get("error") or "")[:400])
        print()

== gemma4-e2b-plecpu F1-001 | 450 token | 44.09s | VRAM 7073.2 MB
```json
[
  {
    "name": "inventory_writedown_ratio_zscore",
    "formula": "zscore(inventory_writedown, inventory_gross, current_assets)",
    "rationale": "Công thức này đo lường mức độ dự phòng giảm giá hàng tồn kho so với giá trị hàng tồn kho gốc và tổng tài sản, chuẩn hóa theo lát cắt ngang. Lợi suất cao có thể cho thấy quản lý rủi ro hàng tồn kho hiệu quả hoặc ngược lại, nếu dự phòng quá ca

== gemma4-e2b-plecpu F2-001 | 378 token | 37.731s | VRAM 7077.2 MB
```json
[
  {
    "name": "inventory_writedown_zscore",
    "formula": "zscore(inventory_writedown)",
    "rationale": "Đo lường mức độ biến động của dự phòng giảm giá hàng tồn kho so với mức trung bình của toàn thị trường, cho thấy rủi ro tiềm ẩn từ việc ghi nhận lỗ trong kỳ.",
    "source": {
      "type": "line_item",
      "ref": "Dự phòng giảm giá hàng tồn kho"
    }
  },
  {
    "name": "inv

== qwen35-0.8b F1-001 | 512 token | 35.335s | VRAM 1501.2 MB
[


In [5]:
# 4. Chạy thật: 2 hàng đợi song song, mỗi card một hàng. Chia sao cho thời gian hai bên gần bằng nhau.
#    Log từng mô hình ở bench/logs/. Ô này in tiến độ 5 phút/lần và chờ tới khi cả hai xong.
import subprocess, time, pathlib

QUEUES = {
    0: [("qwen35-2b", []), ("sailor2-1b", [])],
    1: [("gemma4-e2b-plecpu", []), ("qwen35-0.8b", []),
        ("gemma4-e2b-gpu", ["--seeds", "0", "--limit", "10"])],   # dòng tham chiếu VRAM
}
pathlib.Path("bench/logs").mkdir(exist_ok=True)
script = {gpu: " && ".join(f"python -u bench/run_infer.py --model {k} {' '.join(extra)} > bench/logs/{k}.log 2>&1"
                           for k, extra in q) for gpu, q in QUEUES.items()}
procs = {gpu: subprocess.Popen(["bash", "-c", s], env={**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)})
         for gpu, s in script.items()}

def tail(path, n=1):
    try:
        return open(path, encoding="utf-8").read().strip().splitlines()[-n:]
    except FileNotFoundError:
        return []

t0 = time.time()
while any(p.poll() is None for p in procs.values()):
    time.sleep(300)
    print(f"--- {(time.time() - t0) / 60:.0f} phút")
    for gpu, q in QUEUES.items():
        for k, _ in q:
            last = tail(f"bench/logs/{k}.log")
            if last:
                print(f"  GPU{gpu} {k}: {last[-1][:160]}")
print({gpu: p.returncode for gpu, p in procs.items()}, f"tổng {(time.time() - t0) / 3600:.1f} giờ")

--- 5 phút
  GPU0 qwen35-2b:   seed 0 F4-002  10/228 | JSON ok 80% | 23.841s 14.05 tok/s | VRAM 3721.2 MB | còn ~61 phút
  GPU1 gemma4-e2b-plecpu:    model.embed_audio: -> CPU
--- 10 phút
  GPU0 qwen35-2b:   seed 0 F2-010  30/228 | JSON ok 83% | 14.653s 13.31 tok/s | VRAM 4753.2 MB | còn ~56 phút
  GPU1 gemma4-e2b-plecpu:   seed 0 F4-002  10/228 | JSON ok 90% | 26.98s 9.86 tok/s | VRAM 7081.2 MB | còn ~141 phút
--- 15 phút
  GPU0 qwen35-2b:   seed 0 F3-013  50/228 | JSON ok 76% | 13.578s 13.33 tok/s | VRAM 4755.2 MB | còn ~51 phút
  GPU1 gemma4-e2b-plecpu:   seed 0 F1-006  20/228 | JSON ok 85% | 39.483s 10.03 tok/s | VRAM 7081.2 MB | còn ~136 phút
--- 20 phút
  GPU0 qwen35-2b:   seed 0 F1-014  60/228 | JSON ok 70% | 9.687s 13.63 tok/s | VRAM 4755.2 MB | còn ~50 phút
  GPU1 gemma4-e2b-plecpu:   seed 0 F1-006  20/228 | JSON ok 85% | 39.483s 10.03 tok/s | VRAM 7081.2 MB | còn ~136 phút
--- 25 phút
  GPU0 qwen35-2b:   seed 0 F4-014  70/228 | JSON ok 64% | 9.426s 13.47 tok/s | VRAM 4755.2 M

In [6]:
# 5. Chấm điểm + bảng kết quả + mẫu chấm tay mù (việc 2.6).
!python bench/score.py --c4-sample

# Kết quả benchmark P0

Sinh bởi `bench/score.py` từ `bench/raw`. Định nghĩa chỉ số: `bench/README.md`.

## Bảng chính (đưa vào tờ trình)

| Mô hình | Tham số | VRAM đỉnh card (MB) | VRAM torch (MB) | tok/s | s/công thức hợp lệ | Hợp lệ (%) | Trùng (%) | JSON (%) | Seed | Lượt |
|---|---|---|---|---|---|---|---|---|---|---|
| qwen35-2b | 2B | 5429 (nền 451) | 2685.5 | 13.73 | 49.03 | 50.6 ± 8.4 | 22.7 | 61.4 | 3 | 228 |
| gemma4-e2b-gpu * | E2B (5,1B tổng) | 8761 (nền 451) | 7378.7 | 10.16 | 47.26 | 29.6 | 33.3 | 90.0 | 1 | 10 |
| gemma4-e2b-plecpu | E2B (5,1B tổng) | 7083 (nền 451) | 3054.7 | 9.91 | 27.9 | 51.2 ± 2.1 | 42.5 | 92.1 | 3 | 228 |
| qwen35-0.8b | 0,8B | 4505 (nền 451) | 1757.6 | 14.06 | 77.89 | 21.6 ± 1.7 | 64.1 | 55.7 | 3 | 228 |
| sailor2-1b | 1B | 7257 (nền 451) | 2513.7 | 10.6 | 305.97 | 22.1 ± 7.9 | 9.8 | 31.1 | 3 | 228 |

Hợp lệ = parse được theo DSL ∧ mọi biến có trong danh mục ∧ không trùng (dạng chuẩn hoá). Giá trị `a ± b` = trung bình ± độ lệch chuẩn giữa các see

In [7]:
# 6. Đóng gói để tải về máy (Output -> bench_results.zip). Log thô giữ nguyên, bảng có thể chấm lại ở local.
!cd /kaggle/working/Nico-tick && zip -qr /kaggle/working/bench_results.zip bench/raw bench/logs bench/results.csv bench/table.md bench/formulas.csv eval/c4_blind.csv eval/c4_key.csv
!ls -lh /kaggle/working/bench_results.zip

-rw-r--r-- 1 root root 382K Sep 24 20:04 /kaggle/working/bench_results.zip
